Used to create the config file for the repo extractor (version that accepts multiple repos) to get the comments for the exact prs in human_pull_request from AIdev.

In [1]:
import pandas as pd
import json

In [2]:
human_prs = pd.read_parquet('./data/human_pull_request.parquet')

In [3]:
human_prs.head(10)

,id,number,title,user,user_id,state,created_at,closed_at,merged_at,repo_url,html_url,body,agent
0,2336888723,85268,feat(aci): add automations index page,ameliahsu,55610339,closed,2025-02-14T19:04:59Z,2025-02-18T22:42:20Z,2025-02-18T22:42:19Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85268,https://sentry-j41gpomr5.sentry.dev/automation...,Human
1,2447123365,89131,ref(insights): Make use of `<FeatureBadge>` fo...,ryan953,187460,closed,2025-04-08T23:29:50Z,2025-04-09T15:56:55Z,2025-04-09T15:56:54Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/89131,Using the premade component reduces an import ...,Human
2,2438086945,88748,:bug: fix: update how we fetch workflow_id and...,iamrajjoshi,33237075,closed,2025-04-03T21:36:59Z,2025-04-04T15:10:57Z,2025-04-04T15:10:57Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/88748,i realized i made a mistake for how i fetch th...,Human
3,2265431531,83085,fix(org-stats): Require project membership,ArthurKnaus,7033940,closed,2025-01-08T07:47:13Z,2025-01-08T08:49:40Z,2025-01-08T08:49:40Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/83085,### Problem\r\n\r\nIf the user is not member o...,Human
4,2332333882,85102,ref(consumers): Rename parallel -> batched-par...,evanpurkhiser,1421724,closed,2025-02-12T21:24:17Z,2025-02-12T22:20:33Z,2025-02-12T22:20:33Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85102,Both crons and uptime consumers have a paralle...,Human
5,2486573779,90516,ref(perf-issues): Consolidate File IO override...,leeandher,35509934,closed,2025-04-28T18:17:36Z,2025-04-28T19:22:01Z,2025-04-28T19:22:01Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/90516,This PR removes the `performance_issues.file_i...,Human
6,2474210990,90082,feat(compare): Add timestamp column to sample ...,shruthilayaj,63818634,closed,2025-04-22T17:55:10Z,2025-04-22T21:40:26Z,2025-04-22T21:40:26Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/90082,Include timestamp column in sample mode and ma...,Human
7,2404610772,87421,feat(tsc): Replace SessionsRequest w/ useSessi...,billyvg,79684,closed,2025-03-19T18:07:08Z,2025-03-20T14:17:00Z,2025-03-20T14:17:00Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/87421,Depends on https://github.com/getsentry/sentry...,Human
8,2354374131,85761,ref(crons): Remove time restriction on uptime ...,evanpurkhiser,1421724,closed,2025-02-24T17:58:51Z,2025-02-25T20:18:32Z,2025-02-25T20:18:32Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85761,"There will only ever be one issue, so until we...",Human
9,2564963212,92762,feat(insights): update ux for open in explore ...,DominikB2014,44422760,closed,2025-06-03T18:57:57Z,2025-06-03T20:34:14Z,2025-06-03T20:34:14Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/92762,Updates the open in explore feature and create...,Human


In [4]:
with open('./data/sample_extractor_config.json', 'r') as file:
    config = json.load(file)

In [5]:
print(type(config["targets"]))

<class 'list'>


In [6]:
print(json.dumps(config, indent=4))

{
    "auth_path": "./data/auth/my_auth.txt",
    "output_path": "./data/output/example_output.json",
    "defaults": {
        "state": "closed",
        "labels": [],
        "fields": {
            "issues": [],
            "comments": [
                "body",
                "userid",
                "userlogin",
                "created_at"
            ],
            "commits": []
        }
    },
    "targets": [
        {
            "repo": "facebook/react",
            "range": {
                "start": 1,
                "end": 2
            }
        },
        {
            "repo": "JabRef/jabref",
            "range": {
                "start": 150,
                "end": 151
            }
        }
    ]
}


In [7]:
def create_repo_pr_dict(repo, pr):
    range_ = {"start":pr, "end":pr}
    target = {"repo":repo, "range":range_}
    return target

In [8]:
def create_list_of_prs(prs):
    prs_list = []
    
    #loop through all prs and add them to corresponding repo
    for pr in prs.iterrows():
        pr = pr[1]
        repo = '/'.join(pr["repo_url"].split('/')[-2:])
        pr_num = pr["number"]
        
        target = create_repo_pr_dict(repo, pr_num)
        prs_list.append(target)
    
    return prs_list

In [9]:
prs_targets = create_list_of_prs(human_prs)
config["targets"] = prs_targets
#print(json.dumps(config, indent=4))

In [10]:
with open("./data/human_prs_config.json", "w") as file:
    json.dump(config, file, indent=4)